# 06 Headline validation

**Question.** Which headline numbers in MASTER_RESULTS.csv match the nested-bootstrap, operating-point, volume, and holdout artifacts?

In [ ]:
from pathlib import Path
import os, json, csv
import pandas as pd
import numpy as np

ROOT = Path.cwd()
if not (ROOT / "chapter_a").is_dir():
    for cand in (Path(".."), Path("../.."), Path("../../..")):
        if (cand.resolve() / "chapter_a").is_dir():
            ROOT = cand.resolve()
            break
os.chdir(ROOT)
MASTER = pd.read_csv(ROOT / "chapter_a" / "MASTER_RESULTS.csv")
ANDROCT = ROOT / "abrg" / "output" / "androct_2017"

def row_eq(mask, artifact_auc):
    sub = MASTER.loc[mask]
    assert len(sub) >= 1, mask
    mval = float(sub.iloc[0]["auc_floor"])
    aval = float(artifact_auc)
    assert round(mval, 6) == round(aval, 6), (mval, aval)


In [ ]:
print(MASTER.loc[MASTER.is_headline.astype(str)=="True", ["experiment","detector","auc_floor","ci_low","ci_high","ci_type"]].to_string(index=False))
bias = json.loads((ANDROCT / "ocdev" / "validation" / "check1_bias" / "bias_stats.json").read_text())
d1p = bias["partA_D1_centroid"]["full_sample_point"]
s1m = bias["partB_T1K_S1_norm"]["bootstrap"]["mean"]
row_eq(MASTER.detector=="centroid_euclidean_nested_form", d1p)
row_eq(MASTER.detector=="S1_norm_nested_bootstrap_mean", s1m)
op = json.loads((ANDROCT / "final_validation" / "check2_operating" / "check2.json").read_text())
print(pd.read_csv(ROOT / "chapter_a" / "tables" / "T7_operating_points.csv").head())
c4 = json.loads((ANDROCT / "final_validation" / "check4_benign_holdout" / "check4.json").read_text())
# pooled centroid
pc = c4["centroid_euclidean"]["pooled_oof_raw"]["auc_floor"]
row_eq((MASTER.experiment=="final_validate") & (MASTER.detector=="centroid_euclidean"), pc)
print("ok")
